# Чистовик: константный прогноз

Заготовка, которая уже проходит весь путь: читает `data/`, считает CV по фолдам, пишет `submission.csv` и `cv.json` и печатает `cv.json` последней строкой. Замените прогноз и разбиение на свои.

In [3]:
import json, sys
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

sys.path.insert(0, "data")
from metric_cv import quantile_quality, cv_json

tr = pd.read_csv("data/train.csv")
te = pd.read_csv("data/test.csv")
sample = pd.read_csv("data/sample_submission.csv")
y = tr["daily_orders"].to_numpy(float)
QUANTILES = (0.1, 0.5, 0.9)


In [4]:
tr.head()

,id,city,district_type,latitude,longitude,dist_to_center_km,premise_area_sqm,fitting_rooms_count,entrance_type,floor,parking_nearby,dist_to_metro_m,dist_to_bus_stop_m,residential_units_500m,residential_units_1000m,competitor_pvz_count_500m,commercial_poi_count_500m,avg_housing_price_sqm,nearest_wb_pvz_dist_m,daily_orders
0,PVZ_TR_000001,Самара,center,53.24416,50.17513,3.08,72.5,4,courtyard,-1,1,3122,15,651,1803,1,9,146740.0,444.0,42
1,PVZ_TR_000002,Новосибирск,suburb,54.99110,82.86342,5.00,98.7,6,shopping_mall,1,0,3246,15,490,1268,1,6,144137.0,389.0,49
2,PVZ_TR_000003,Москва,industrial,55.58814,37.47537,20.66,62.3,3,courtyard,-1,1,5000,129,131,368,0,4,277898.0,802.0,25
3,PVZ_TR_000004,Москва,industrial,55.59576,37.45754,20.42,71.7,4,shopping_mall,1,1,5000,377,160,404,1,0,175257.0,1072.0,25
4,PVZ_TR_000005,Екатеринбург,suburb,56.90393,60.50738,9.38,90.8,5,courtyard,-1,1,5000,36,394,1133,1,8,125757.0,250.0,25


In [7]:
def predict(y_train, n):
    """Константный прогноз: квантили спроса по всему train. Замените на модель."""
    q = np.quantile(y_train, QUANTILES)
    print(q)
    return np.tile(q, (n, 1))

In [8]:
# CV: четыре случайных фолда — замените разбиением, которое похоже на тест
splits = list(KFold(4, shuffle=True, random_state=0).split(tr))
oof = np.zeros((len(tr), 3))
for a, b in splits:
    oof[b] = predict(y[a], len(b))
    print(predict(y[a], len(b)))
folds = [100 * quantile_quality(y[b], oof[b, 0], oof[b, 1], oof[b, 2], tr["city"].iloc[b]) for _, b in splits]

# посылка: колонки и строки как в sample_submission.csv, 0 <= p10 <= p50 <= p90 <= 5000
test = np.round(predict(y, len(te)), 1)
sub = pd.DataFrame({"id": te["id"],
                    "predicted_daily_orders_p10": test[:, 0],
                    "predicted_daily_orders_p50": test[:, 1],
                    "predicted_daily_orders_p90": test[:, 2]})
assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample)
assert set(sub["id"]) == set(sample["id"]) and (sub.iloc[:, 1:].to_numpy() >= 0).all()
sub.to_csv("submission.csv", index=False)

cv = cv_json("constant", [f / 100 for f in folds])   # пишет cv.json
print(json.dumps(cv))                                  # последняя строка — для проверки воспроизводимости

[ 25.  48. 217.]
[ 25.  48. 217.]
[[ 25.  48. 217.]
 [ 25.  48. 217.]
 [ 25.  48. 217.]
 ...
 [ 25.  48. 217.]
 [ 25.  48. 217.]
 [ 25.  48. 217.]]
[ 25.   48.  221.2]
[ 25.   48.  221.2]
[[ 25.   48.  221.2]
 [ 25.   48.  221.2]
 [ 25.   48.  221.2]
 ...
 [ 25.   48.  221.2]
 [ 25.   48.  221.2]
 [ 25.   48.  221.2]]
[ 25.  49. 232.]
[ 25.  49. 232.]
[[ 25.  49. 232.]
 [ 25.  49. 232.]
 [ 25.  49. 232.]
 ...
 [ 25.  49. 232.]
 [ 25.  49. 232.]
 [ 25.  49. 232.]]
[ 25.  47. 215.]
[ 25.  47. 215.]
[[ 25.  47. 215.]
 [ 25.  47. 215.]
 [ 25.  47. 215.]
 ...
 [ 25.  47. 215.]
 [ 25.  47. 215.]
 [ 25.  47. 215.]]
[ 25.  48. 220.]
{"name": "constant", "metric": "quantile_quality_x100", "folds": [0.0, 0.0, 0.0, 0.0], "cv": 0.0, "std": 0.0}
